Select a few samples from each fold to test the model's performance on extreme low-data scenarios.

In [40]:
import pandas as pd
import rasterio as rio
from glob import glob
import os
import numpy as np
from mslandcover.config import LEGEND_CLASSES
from shutil import copyfile

n_samples_per_class_per_split = 4
sampled_data = {}
for split_idx in range(4):
    
    split_paths = glob(f'../data/splits/split_{split_idx+1}/target/*.tif')
    
    histograms = []
    for file_path in split_paths:
        file_id = os.path.basename(file_path).split('.')[0]
        
        data = rio.open(file_path).read()
        histogram = np.histogram(data, bins=8, range=(1,9))[0]
        
        record = {'id': file_id}
        for k, v in LEGEND_CLASSES.items():
            if k == 0:
                continue
            record[v] = histogram[k-1].item()
        
        histograms.append(record)
    
    df = pd.DataFrame.from_records(histograms)
    sampled_data[split_idx + 1] = []
    
    for i in range(n_samples_per_class_per_split):
        for col in df.columns:
            if col in ['Unclassified', 'id']:
                continue
            
            df = df.sort_values(by=col, ascending=False)
            sampled_data[split_idx + 1].append(df.iloc[0].id)
            
            df = df.drop(df.iloc[0].name)

In [42]:
for split_idx in sampled_data.keys():
    print(len(sampled_data[split_idx]))
    
    os.makedirs(f'../data/small_splits/split_{split_idx}/input/', exist_ok=True)
    os.makedirs(f'../data/small_splits/split_{split_idx}/target/', exist_ok=True)
    
    for file_id in sampled_data[split_idx]:
        
        copyfile(
            f'../data/splits/split_{split_idx}/input/{file_id}.tif', 
            f'../data/small_splits/split_{split_idx}/input/{file_id}.tif'
        )
        copyfile(
            f'../data/splits/split_{split_idx}/target/{file_id}.tif', 
            f'../data/small_splits/split_{split_idx}/target/{file_id}.tif'
        )

28
28
28
28
